# 01 — EDA и предобработка данных

> **Предполагается, что ноутбук `00_dataset_collection.ipynb` уже запущен** и сохранил `processed_data/` + `label_names.json` в `/kaggle/working/`.
> Если нет — этот ноутбук загрузит данные самостоятельно.

**Что делаем:**
1. Загружаем объединённый датасет (~93k примеров)
2. Разведочный анализ по всем 7 эмоциям
3. Анализ длин текстов
4. Примеры текстов по классам

In [ ]:
import sys, os

PROJECT_ROOT = "/kaggle/input/sentiment-analysis" if os.path.exists("/kaggle") else os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

WORKING_DIR = "/kaggle/working" if os.path.exists("/kaggle") else "../results"
os.makedirs(WORKING_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings
warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
print(f'Working dir: {WORKING_DIR}')

## 1. Загрузка данных

In [ ]:
from datasets import load_from_disk
from src.data_loader import EKMAN_LABEL_NAMES, EKMAN_ID2LABEL

DATA_PATH = f'{WORKING_DIR}/processed_data'
LABEL_NAMES_PATH = f'{WORKING_DIR}/label_names.json'

if os.path.exists(DATA_PATH) and os.path.exists(LABEL_NAMES_PATH):
    dataset = load_from_disk(DATA_PATH)
    with open(LABEL_NAMES_PATH) as f:
        label_names = json.load(f)
    print(f'Загружен объединённый датасет из: {DATA_PATH}')
else:
    print('Датасет не найден, запускаю сборку автоматически...')
    from src.data_loader import load_data
    from src.preprocessor import preprocess_batch
    dataset, label_names, _ = load_data(use_all=True)
    dataset = dataset.map(lambda b: preprocess_batch(b, clean=True), batched=True, desc='Cleaning')
    dataset.save_to_disk(DATA_PATH)
    with open(LABEL_NAMES_PATH, 'w') as f:
        json.dump(label_names, f)

print(f'Классы: {label_names}')
for split in dataset:
    print(f'  {split:12s}: {len(dataset[split]):,}')

In [ ]:
train_df = dataset['train'].to_pandas()
val_df   = dataset['validation'].to_pandas()
test_df  = dataset['test'].to_pandas()
all_df   = pd.concat([train_df, val_df, test_df], ignore_index=True)
all_df['emotion'] = all_df['label'].map(EKMAN_ID2LABEL)
print(f'Всего примеров: {len(all_df):,}')
all_df.head()

## 2. Распределение эмоций

In [ ]:
EMOTION_COLORS = {
    'anger': '#e74c3c', 'disgust': '#8e44ad', 'fear': '#2c3e50',
    'joy': '#f39c12', 'sadness': '#3498db', 'surprise': '#1abc9c', 'neutral': '#95a5a6',
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

counts = all_df['emotion'].value_counts().reindex(label_names).fillna(0)
colors = [EMOTION_COLORS.get(e, '#999') for e in counts.index]
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white')
axes[0].set_title(f'Распределение классов (всего {len(all_df):,})')
for bar, v in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{int(v):,}\n({v/len(all_df)*100:.1f}%)', ha='center', fontsize=8)

split_data = {}
for split, df in [('train', train_df), ('val', val_df), ('test', test_df)]:
    split_data[split] = df['label'].map(EKMAN_ID2LABEL).value_counts(normalize=True).reindex(label_names).fillna(0)
pd.DataFrame(split_data).plot(kind='bar', ax=axes[1], color=['#3498db','#e67e22','#9b59b6'])
axes[1].set_title('Распределение по сплитам (доли)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/emotion_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Длины текстов по классам
all_df['word_count'] = all_df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

data_by_emotion = [all_df[all_df['emotion'] == e]['word_count'].dropna().values for e in label_names]
bp = axes[0].boxplot(data_by_emotion, labels=label_names, patch_artist=True)
for patch, e in zip(bp['boxes'], label_names):
    patch.set_facecolor(EMOTION_COLORS.get(e, '#ccc'))
axes[0].set_title('Длина текстов по эмоциям (слова)')
axes[0].tick_params(axis='x', rotation=30)

axes[1].hist(all_df['word_count'].clip(upper=100), bins=50, color='#3498db', alpha=0.7, edgecolor='white')
med = all_df['word_count'].median()
axes[1].axvline(med, color='red', linestyle='--', label=f'Медиана: {med:.0f}')
axes[1].set_title('Гистограмма длин (обрезано до 100 слов)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/text_lengths.png', dpi=150, bbox_inches='tight')
plt.show()
print(all_df[['word_count']].describe().round(1))

In [ ]:
# Примеры по каждой эмоции
for emotion in label_names:
    print(f'\n{"═"*60}  {emotion.upper()}')
    samples = all_df[all_df['emotion'] == emotion]['text'].sample(
        min(2, (all_df['emotion'] == emotion).sum()), random_state=42
    )
    for i, text in enumerate(samples, 1):
        print(f'  {i}. {str(text)[:180]}')

In [ ]:
print('\n' + '='*60)
print('ИТОГО')
print('='*60)
for split in ['train', 'validation', 'test']:
    print(f'{split:12s}: {len(dataset[split]):,}')
print(f'Классы: {label_names}')